In [ ]:
import geopandas as gpd
import networkx as nx
import osmnx as ox
import pandas as pd
import shapely
from shapely.geometry import MultiPolygon, Polygon
from tqdm.notebook import tqdm

from paths import paths, prepare_preprocessing
from preprocessing.src.constants import CRS_GEO, CRS_METRIC, LSOA_IDS

In [ ]:
ox.settings.cache_folder = paths.preprocessing.cache

In [ ]:
prepare_preprocessing()

In [ ]:
NEIGHBOURHOOD_MINUTES = (10, 15, 20)
TRIP_TIMES = [5, 10, 15, 20]
BUFFER_DEFAULT = 25
WALK_SPEED_MPS = 4.8 * 1000 / 3600
INTERSECTION_PENALTY = 1.6

SUBGRAPH_PATH = paths.preprocessing.shared.subgraphs
ISOCHRONE_PATH = paths.preprocessing.shared.isochrones

In [ ]:
# Helper function to remove holes from polygons
def remove_holes(geom):
    if geom.is_empty:
        return geom

    if geom.geom_type == 'Polygon':
        return Polygon(geom.exterior)

    if geom.geom_type == 'MultiPolygon':
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])

    return geom

In [ ]:
# Calculate edge travel time according to length and walking speed
def attach_travel_time(G):
    for _, _, data in G.edges(data = True):
        data['travel_time'] = data['length'] / WALK_SPEED_MPS

    return G

In [ ]:
# Build the neighbourhood isochrones using the walkable graph
def build_isochrones(G, center_node, trip_times_min, edge_buff):
    edge_geoms = {}
    buffer_cache = {}
    seen_edges = set()
    prev_iso = None

    rows = []

    # Add intersection penalty if the intersection degree is greater than 2
    weight_func = lambda _, v, data: (
        list(data.values())[0]['travel_time'] + (INTERSECTION_PENALTY if G.degree(v) > 2 else 0)
    )

    for minutes in trip_times_min:
        cutoff = minutes * 60

        # Compute reachable nodes within cutoff time
        reach = nx.single_source_dijkstra_path_length(G, center_node, cutoff = cutoff, weight = weight_func)

        sub = G.subgraph(reach.keys()).copy()

        if not sub.number_of_edges():
            rows.append((minutes, Polygon()))
            continue

        # Get edge geometries for the subgraph
        edges_gdf = ox.graph_to_gdfs(sub, nodes = False, edges = True)
        edge_lookup = {(u, v, key): geom for (u, v, key), geom in zip(edges_gdf.index, edges_gdf.geometry)}

        current_edge_ids = set(sub.edges(keys = True))
        new_edge_ids = current_edge_ids - seen_edges

        if not new_edge_ids:
            rows.append((minutes, prev_iso))
            continue

        buffered_pieces = []
        for edge_id in new_edge_ids:
            # Cache raw edge geometries
            if edge_id not in edge_geoms:
                edge_geoms[edge_id] = edge_lookup[edge_id]

            # Cache buffered geometries
            if edge_id not in buffer_cache:
                buffer_cache[edge_id] = edge_geoms[edge_id].buffer(edge_buff)

            buffered_pieces.append(buffer_cache[edge_id])

        # Merge new buffered pieces into one geometry and simplify
        new_buffers = shapely.union_all(buffered_pieces)
        new_buffers = new_buffers.simplify(tolerance = minutes / 5, preserve_topology = True)

        # Accumulate into the running isochrone
        if prev_iso is None:
            prev_iso = new_buffers
        else:
            prev_iso = prev_iso.union(new_buffers)

        prev_iso = remove_holes(prev_iso)

        seen_edges.update(new_edge_ids)
        rows.append((minutes, prev_iso))

    # Build GDF of cumulative isochrones
    gdf = gpd.GeoDataFrame({'minutes': [m for m, _ in rows], 'geometry': [poly for _, poly in rows]}, crs = CRS_METRIC)
    gdf = gdf.sort_values('minutes').reset_index(drop = True)

    # Convert cumulative isochrones into rings
    prev_poly = None

    for i, geom in enumerate(gdf.geometry):
        if prev_poly is not None:
            gdf.at[i, 'geometry'] = geom.difference(prev_poly)

        prev_poly = geom

    return gdf

In [ ]:
# Build isochrones for all seed nodes for a given LSOA
def build_lsoa_isochrones(G, seed_nodes, trip_times_min, edge_buff = BUFFER_DEFAULT):
    rows = []

    for seed_node in seed_nodes:
        isochrones = build_isochrones(G, seed_node, trip_times_min, edge_buff)
        isochrones['seed_node'] = seed_node
        rows.append(isochrones)

    return pd.concat(rows, ignore_index = True)

In [ ]:
# Load the LSOAs and walkable graph, then attach travel times
lsoas = gpd.read_parquet(paths.preprocessing.shared.lsoa)

G = ox.load_graphml(paths.preprocessing.shared.network)
G = attach_travel_time(G)

In [ ]:
# Extract seed nodes from LSOAs
seed_nodes = lsoas[[f'sn{i}' for i in range(10)]].map(lambda x: None if pd.isna(x) else int(x))

In [ ]:
# Reuse cached rings only when all current seeds and durations are present
def has_complete_isochrones(output, lsoa_id, seed_nodes):
    if not output.is_file():
        return False

    try:
        cached = gpd.read_file(output)
    except Exception as e:
        print(f'Rebuilding unreadable isochrone cache for {lsoa_id}: {e}')

        return False

    required = {'seed_node', 'minutes', 'LSOA21CD', 'geometry'}

    if not required.issubset(cached.columns) or cached.crs != CRS_METRIC:
        return False

    if cached[['seed_node', 'minutes', 'LSOA21CD']].isna().any().any() or not cached['LSOA21CD'].eq(lsoa_id).all():
        return False

    coverage = cached.groupby('seed_node')['minutes'].agg(set)

    return (
        not coverage.empty
        and set(coverage.index) == set(seed_nodes)
        and all(set(TRIP_TIMES).issubset(minutes) for minutes in coverage)
    )

def calculate_isochrones(lsoa_id, seed_nodes):
    output = ISOCHRONE_PATH / f'{lsoa_id}.geojson'

    if has_complete_isochrones(output, lsoa_id, seed_nodes):
        return

    try:
        isochrones = build_lsoa_isochrones(G, seed_nodes, TRIP_TIMES)
        isochrones['LSOA21CD'] = lsoa_id
        isochrones.to_file(output, driver = 'GeoJSON')
    except Exception as e:
        raise RuntimeError(f'Error processing {lsoa_id}') from e

for lsoa_id, snodes in tqdm(zip(LSOA_IDS, seed_nodes.values.tolist()), total = len(LSOA_IDS), desc = 'Isochrones'):
    calculate_isochrones(lsoa_id, snodes)

In [ ]:
# Export all required neighbourhood durations from each LSOA's shared rings
def generate_neighbourhoods(lsoa_id):
    gdf = gpd.read_file(ISOCHRONE_PATH / f'{lsoa_id}.geojson')
    coverage = gdf.groupby('seed_node')['minutes'].agg(set)

    if coverage.empty or any(not set(TRIP_TIMES).issubset(values) for values in coverage):
        raise ValueError(f'{lsoa_id} is missing required isochrone rings.')

    for minutes in NEIGHBOURHOOD_MINUTES:
        selected = gdf[gdf['minutes'] <= minutes]
        unified = {seed: group.geometry.union_all() for seed, group in selected.groupby('seed_node')}

        neighbourhoods = gpd.GeoDataFrame(
            {'seed_node': list(unified), 'geometry': list(unified.values())}, crs = gdf.crs
        ).to_crs(CRS_GEO)

        output = paths.preprocessing.walkability[minutes].neighborhoods
        neighbourhoods.to_file(output / f'{lsoa_id}.geojson', driver = 'GeoJSON')

for lsoa_id in tqdm(LSOA_IDS, desc = 'Neighbourhoods'):
    generate_neighbourhoods(lsoa_id)

In [ ]:
LSOA = 'E01000001'

gdf = gpd.read_file(ISOCHRONE_PATH / f'{LSOA}.geojson')
gdf = gdf[gdf.seed_node == gdf.seed_node.iloc[0]]

gdf.explore(
    column = 'minutes',
    cmap = 'viridis_r',
    legend = True,
    tooltip = True,
    name = 'Isochrones',
    tiles = 'CartoDB positron',
    style_kwds = {'weight': 0.5, 'lineOpacity': 0.5, 'fillOpacity': 0.5},
)